# H-08: Tail Risk Analysis

**Question:** Are the worst-case outcomes (tail events) disproportionately severe compared to typical performance?

| | |
|---|---|
| **H₀ (Null):** | Tail outcomes are not disproportionately severe. |
| **Hₐ (Alternative):** | The worst 5% average significantly worse than P95, indicating hidden catastrophic risk. |
| **Certification rule:** | Per-sub-fault CVaR/IQM ratio determines risk level; category risk = worst sub-fault. |
| **Metrics:** | time_to_detect, time_to_mitigate |
| **Method:** | CVaR (Conditional Value-at-Risk) at the 95th percentile |

### Risk Levels (CVaR/IQM ratio)
| Ratio | Level | Meaning |
|-------|-------|---------|
| ratio < 1.5 | ✅ mild | Tail outcomes close to typical — acceptable |
| 1.5 ≤ ratio < 2.0 | ⚠️ moderate | Tail outcomes noticeably worse — investigate |
| ratio ≥ 2.0 | ❌ significant | Catastrophic tail risk — action required |

### Key Properties
- **Always active** — does not require SLA thresholds. SLA thresholds are optional and only used for overshoot computation.
- **Detected-only** — uses only runs where the fault was detected (not all runs).
- **CVaR** = mean of values beyond the P95 quantile (expected shortfall).
- **IQM** = interquartile mean (robust central tendency, trimming bottom 25% and top 25%).


In [1]:
import sys, os, json, yaml, numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"
gt_root = project_root / "hypothesis_framework" / "data" / "groundtruth" / "kubernetes"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True):
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        val = q.get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

def load_sla_thresholds(gt_dir, metric_key):
    thresholds = {}
    for fault_dir in sorted(gt_dir.iterdir()):
        gt_file = fault_dir / "ground_truth.yaml"
        if not gt_file.exists():
            continue
        with open(gt_file, "r", encoding="utf-8") as f:
            gt = yaml.safe_load(f)
        sla = gt.get("ground_truth", {}).get("sla", {})
        metric_sla = sla.get(metric_key, {})
        if "threshold" in metric_sla:
            thresholds[fault_dir.name] = float(metric_sla["threshold"])
    return thresholds

# Load runs
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

# Load SLA thresholds (optional for H-08)
ttd_sla = load_sla_thresholds(gt_root, "time_to_detect")
ttm_sla = load_sla_thresholds(gt_root, "time_to_mitigate")

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r['quantitative'].get('fault_detected') == 'Yes')
    faults = sorted(set(r['fault_name'] for r in runs))
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in faults:
        fn_det = sum(1 for r in runs if r['fault_name'] == fn and r['quantitative'].get('fault_detected') == 'Yes')
        sla_ttd = ttd_sla.get(fn)
        sla_ttm = ttm_sla.get(fn)
        sla_ttd_str = f"{sla_ttd:.0f}" if sla_ttd is not None else "\u2014"
        sla_ttm_str = f"{sla_ttm:.0f}" if sla_ttm is not None else "\u2014"
        print(f"    {fn}: {fn_det} detected  (SLA: TTD={sla_ttd_str}s, TTM={sla_ttm_str}s)")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 28 detected  (SLA: TTD=—s, TTM=—s)
    pod-delete: 23 detected  (SLA: TTD=60s, TTM=120s)
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 13 detected  (SLA: TTD=60s, TTM=180s)
    pod-network-corruption: 14 detected  (SLA: TTD=240s, TTM=420s)
    pod-network-loss: 12 detected  (SLA: TTD=180s, TTM=360s)
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 25 detected  (SLA: TTD=60s, TTM=180s)
    pod-cpu-hog: 25 detected  (SLA: TTD=120s, TTM=300s)
    pod-memory-hog: 20 detected  (SLA: TTD=90s, TTM=240s)


---
## Step 1: Run H-08 — time_to_detect

Computes CVaR at the 95th percentile for each sub-fault using detected-only runs.
Risk level is determined by the CVaR/IQM ratio. SLA thresholds (optional) enable overshoot analysis.


In [2]:
from hypothesis_framework.scripts.hypothesis_tests.h08_tail_risk_analysis import run_tail_risk_test

ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect")

h08_ttd = run_tail_risk_test(
    data_per_category=ttd_data,
    metric_name="time_to_detect",
    quantile=0.95,
    sla_thresholds=ttd_sla,
)

print(f"H-08: {h08_ttd.hypothesis_name}")
print(f"Metric: {h08_ttd.metric_name}  |  Quantile: {h08_ttd.quantile_level}")
print(f"Overall: {h08_ttd.overall_assessment}")
print("=" * 100)

RISK_ICONS = {"mild": "\u2705", "moderate": "\u26a0\ufe0f", "significant": "\u274c", "unknown": "\u2753"}

for c in h08_ttd.per_category:
    c_icon = RISK_ICONS.get(c.risk_level, "\u2753")
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Risk: {c_icon} {c.risk_level}")
    if c.worst_sub_fault:
        print(f"    Worst sub-fault: {c.worst_sub_fault}")
    print(f"    {'Sub-Fault':25s} {'n':>4s} {'P95':>8s} {'CVaR':>8s} {'n_tail':>6s} {'Ratio':>7s} {'SLA':>8s} {'Overshoot':>10s} {'Risk':>12s}")
    print(f"    {'-' * 92}")
    for sf in c.sub_faults:
        sf_icon = RISK_ICONS.get(sf.risk_level, "\u2753")
        ratio_str = f"{sf.cvar_iqm_ratio:.2f}" if sf.cvar_iqm_ratio is not None else "\u2014"
        sla_str = f"{sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "\u2014"
        overshoot_str = f"{sf.expected_overshoot:.1f}s" if sf.expected_overshoot is not None else "\u2014"
        print(f"    {sf.fault_name:25s} {sf.n:4d} {sf.p95:8.1f} {sf.cvar:8.1f} {sf.n_tail:6d} {ratio_str:>7s} {sla_str:>8s} {overshoot_str:>10s} {sf_icon} {sf.risk_level}")

if h08_ttd.warnings:
    print(f"\nWarnings:")
    for w in h08_ttd.warnings:
        print(f"  - {w}")


H-08: Tail Risk Analysis
Metric: time_to_detect  |  Quantile: 0.95
Overall: significant_tail_risk

  application_fault (n=51, 2 sub-faults):
    Risk: ⚠️ moderate
    Worst sub-fault: container-kill
    Sub-Fault                    n      P95     CVaR n_tail   Ratio      SLA  Overshoot         Risk
    --------------------------------------------------------------------------------------------
    container-kill              28    156.9    195.2      1    1.59        —          — ⚠️ moderate
    pod-delete                  23    193.8    197.4      1    1.50      60s      72.6s ⚠️ moderate

  network_fault (n=39, 3 sub-faults):
    Risk: ❌ significant
    Worst sub-fault: pod-dns-error
    Sub-Fault                    n      P95     CVaR n_tail   Ratio      SLA  Overshoot         Risk
    --------------------------------------------------------------------------------------------
    pod-dns-error               13    662.5    782.9      1   11.19      60s     302.6s ❌ significant
    p

---
## Step 2: Run H-08 — time_to_mitigate

Same tail risk analysis for time-to-mitigate.


In [3]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate")

h08_ttm = run_tail_risk_test(
    data_per_category=ttm_data,
    metric_name="time_to_mitigate",
    quantile=0.95,
    sla_thresholds=ttm_sla,
)

print(f"H-08: {h08_ttm.metric_name}  |  {h08_ttm.overall_assessment}")
print("=" * 100)
for c in h08_ttm.per_category:
    c_icon = RISK_ICONS.get(c.risk_level, "\u2753")
    print(f"  {c.category:20s}: {c_icon} {c.risk_level:12s}  (worst: {c.worst_sub_fault})")
    for sf in c.sub_faults:
        sf_icon = RISK_ICONS.get(sf.risk_level, "\u2753")
        ratio_str = f"{sf.cvar_iqm_ratio:.2f}" if sf.cvar_iqm_ratio is not None else "\u2014"
        sla_str = f"{sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "\u2014"
        overshoot_str = f"{sf.expected_overshoot:.1f}s" if sf.expected_overshoot is not None else "\u2014"
        print(f"    {sf.fault_name:25s}: CVaR={sf.cvar:7.1f}  ratio={ratio_str:>5s}  SLA={sla_str:>6s}  overshoot={overshoot_str:>8s}  {sf_icon} {sf.risk_level}")

if h08_ttm.warnings:
    print(f"\nWarnings: {h08_ttm.warnings}")


H-08: time_to_mitigate  |  significant_tail_risk
  application_fault   : ⚠️ moderate      (worst: pod-delete)
    container-kill           : CVaR=  458.4  ratio= 1.53  SLA=     —  overshoot=       —  ⚠️ moderate
    pod-delete               : CVaR=  511.4  ratio= 1.57  SLA=  120s  overshoot=  210.0s  ⚠️ moderate
  network_fault       : ❌ significant   (worst: pod-dns-error)
    pod-dns-error            : CVaR= 1128.7  ratio= 3.47  SLA=  180s  overshoot=  271.7s  ❌ significant
    pod-network-corruption   : CVaR= 1263.9  ratio= 2.38  SLA=  420s  overshoot=  420.6s  ❌ significant
    pod-network-loss         : CVaR= 1056.1  ratio= 2.28  SLA=  360s  overshoot=  332.4s  ❌ significant
  resource_fault      : ⚠️ moderate      (worst: disk-fill)
    disk-fill                : CVaR=  702.6  ratio= 1.60  SLA=  180s  overshoot=  276.2s  ⚠️ moderate
    pod-cpu-hog              : CVaR=  724.1  ratio= 1.51  SLA=  300s  overshoot=  205.0s  ⚠️ moderate
    pod-memory-hog           : CVaR=  884.5  ra

---
## Step 3: Summary & Interpretation


In [4]:
print("H-08 Tail Risk Analysis \u2014 Summary")
print("=" * 90)

results = {
    "time_to_detect": h08_ttd,
    "time_to_mitigate": h08_ttm,
}

for metric, r in results.items():
    overall_icon = "\u2705" if r.overall_assessment == "acceptable_tail_risk" else "\u274c"
    print(f"\n  {metric}: {overall_icon} {r.overall_assessment}")
    for c in r.per_category:
        c_icon = RISK_ICONS.get(c.risk_level, "\u2753")
        n_mild = sum(1 for sf in c.sub_faults if sf.risk_level == "mild")
        n_mod = sum(1 for sf in c.sub_faults if sf.risk_level == "moderate")
        n_sig = sum(1 for sf in c.sub_faults if sf.risk_level == "significant")
        print(f"    {c.category:20s}: {c_icon} {c.risk_level:12s}  "
              f"({n_mild} mild, {n_mod} moderate, {n_sig} significant)")

print("\n" + "=" * 90)
print("\nInterpretation:")
print("  - CVaR = mean of values exceeding the P95 quantile (expected shortfall).")
print("  - IQM = interquartile mean (robust central tendency).")
print("  - CVaR/IQM ratio measures how extreme the tail is relative to typical performance.")
print("  - mild (< 1.5): tail outcomes are close to the center \u2014 acceptable.")
print("  - moderate (1.5\u20132.0): tail outcomes noticeably worse \u2014 investigate.")
print("  - significant (\u2265 2.0): catastrophic tail risk \u2014 action required.")
print("  - SLA overshoot (optional): expected amount by which tail events exceed the SLA.")


H-08 Tail Risk Analysis — Summary

  time_to_detect: ❌ significant_tail_risk
    application_fault   : ⚠️ moderate      (0 mild, 2 moderate, 0 significant)
    network_fault       : ❌ significant   (0 mild, 0 moderate, 3 significant)
    resource_fault      : ⚠️ moderate      (2 mild, 1 moderate, 0 significant)

  time_to_mitigate: ❌ significant_tail_risk
    application_fault   : ⚠️ moderate      (0 mild, 2 moderate, 0 significant)
    network_fault       : ❌ significant   (0 mild, 0 moderate, 3 significant)
    resource_fault      : ⚠️ moderate      (0 mild, 3 moderate, 0 significant)


Interpretation:
  - CVaR = mean of values exceeding the P95 quantile (expected shortfall).
  - IQM = interquartile mean (robust central tendency).
  - CVaR/IQM ratio measures how extreme the tail is relative to typical performance.
  - mild (< 1.5): tail outcomes are close to the center — acceptable.
  - moderate (1.5–2.0): tail outcomes noticeably worse — investigate.
  - significant (≥ 2.0): catastr

---
## Step 4: JSON Output


In [5]:
output = {
    "hypothesis_id": "H-08",
    "hypothesis_name": "Tail Risk Analysis",
    "null_hypothesis": h08_ttd.null_hypothesis,
    "alt_hypothesis": h08_ttd.alt_hypothesis,
    "certification_rule": "Per-sub-fault CVaR/IQM ratio determines risk level; category risk = worst sub-fault.",
    "hypothesis_metadata": {
        "method": "CVaR (Conditional Value-at-Risk)",
        "quantile": h08_ttd.quantile_level,
        "risk_thresholds": {
            "mild": "CVaR/IQM < 1.5",
            "moderate": "1.5 <= CVaR/IQM < 2.0",
            "significant": "CVaR/IQM >= 2.0",
        },
        "sla_thresholds": {
            "time_to_detect": ttd_sla,
            "time_to_mitigate": ttm_sla,
        },
        "categories": list(categories.keys()),
        "mode": "Always active (SLA thresholds optional for overshoot only)",
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "per_category": [
            {
                "category": c.category,
                "n": c.n,
                "n_sub_faults": c.n_sub_faults,
                "risk_level": c.risk_level,
                "worst_sub_fault": c.worst_sub_fault,
                "sub_faults": [
                    {
                        "fault_name": sf.fault_name,
                        "n": sf.n,
                        "p95": sf.p95,
                        "cvar": sf.cvar,
                        "n_tail": sf.n_tail,
                        "cvar_iqm_ratio": sf.cvar_iqm_ratio,
                        "sla_threshold": sf.sla_threshold,
                        "expected_overshoot": sf.expected_overshoot,
                        "n_breaches": sf.n_breaches,
                        "risk_level": sf.risk_level,
                    }
                    for sf in c.sub_faults
                ],
            }
            for c in r.per_category
        ],
        "warnings": r.warnings,
    }

print(json.dumps(output, indent=2, default=str, ensure_ascii=False))


{
  "hypothesis_id": "H-08",
  "hypothesis_name": "Tail Risk Analysis",
  "null_hypothesis": "Tail outcomes are not disproportionately severe.",
  "alt_hypothesis": "The worst 5% average significantly worse than P95, indicating hidden catastrophic risk.",
  "certification_rule": "Per-sub-fault CVaR/IQM ratio determines risk level; category risk = worst sub-fault.",
  "hypothesis_metadata": {
    "method": "CVaR (Conditional Value-at-Risk)",
    "quantile": 0.95,
    "risk_thresholds": {
      "mild": "CVaR/IQM < 1.5",
      "moderate": "1.5 <= CVaR/IQM < 2.0",
      "significant": "CVaR/IQM >= 2.0"
    },
    "sla_thresholds": {
      "time_to_detect": {
        "disk-fill": 60.0,
        "node-restart": 30.0,
        "pod-autoscaler": 120.0,
        "pod-cpu-hog": 120.0,
        "pod-delete": 60.0,
        "pod-dns-error": 60.0,
        "pod-memory-hog": 90.0,
        "pod-network-corruption": 240.0,
        "pod-network-loss": 180.0,
        "pod-network-rate-limit": 180.0
      },
 